In [12]:
import pandas as pd
import scipy.stats as st
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

ModuleNotFoundError: No module named 'torch'

In [8]:
# Combine 2024 and 2025 monthly signals
signals = pd.concat([exploitSignals, exploitSignals2], ignore_index=True)

# Parse KEV date_added to define your training ground truth
knownExploits["date_added"] = pd.to_datetime(knownExploits["date_added"])
kev_dates = dict(zip(knownExploits["cve_id"], knownExploits["date_added"]))

In [9]:
# Convert signal month (YYYY-MM) to month-end timestamp
signals["month_end"] = pd.to_datetime(signals["month"]) + pd.offsets.MonthEnd(1)

# Map KEV addition date
signals["kev_date"] = signals["cve_id"].map(kev_dates)

# Filter out rows where the CVE was ALREADY on KEV on or before the snapshot date
signals = signals[signals["in_kev"] == 0].copy()

# Label = 1 if added to KEV within 90 days after this snapshot month
signals["days_until_kev"] = (signals["kev_date"] - signals["month_end"]).dt.days
signals["target"] = (
    (signals["days_until_kev"] > 0) & (signals["days_until_kev"] <= 90)
).astype(int)

print("Positive samples in training window:", signals["target"].sum())

Positive samples in training window: 533


In [10]:
# Safe static columns available from the initial disclosure
safe_vul_cols = [
    "cve_id",
    "description",
    "assigner",
    "cwe_ids",
    "attack_vector",
    "attack_complexity",
    "privileges_required",
    "user_interaction",
    "cvss3_score",
]

# Keep only columns present in the dataset
available_cols = [c for c in safe_vul_cols if c in vul.columns]
vul_clean = vul[available_cols].copy()

# Merge with training signals
train_df = signals.merge(vul_clean, on="cve_id", how="left")

# Merge with 2026 test scoreboard
test_df = scoreBoard.merge(vul_clean, on="cve_id", how="left")

In [ ]:
# Load IBM Granite Embedding model from Hugging Face
granite_embedder = SentenceTransformer("ibm-granite/granite-embedding-125m-english")

# Fill missing descriptions
train_df["description"] = train_df["description"].fillna("")
test_df["description"] = test_df["description"].fillna("")

# Compute embeddings (or extract embeddings for unique CVEs to save memory)
unique_cves = pd.concat(
    [
        train_df[["cve_id", "description"]],
        test_df[["cve_id", "description"]],
    ]
).drop_duplicates("cve_id")

embeddings = granite_embedder.encode(
    unique_cves["description"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

embedding_dict = dict(zip(unique_cves["cve_id"], list(embeddings)))

ModuleNotFoundError: No module named 'torch'

In [ ]:
# Example: sort by model prediction score descending
test_df["prediction_score"] = model_scores  # Replace with your model output

top500 = test_df.sort_values(by="prediction_score", ascending=False).head(500)
top500[["cve_id"]].to_csv("top500.csv", index=False)

# Validate output format
assert top500[["cve_id"]].shape == (500, 1)
print("Saved top500.csv successfully.")